# Datasets

> The two public datasets every notebook on this site uses: one household's electricity meter, and the 414 hourly series of the M4 competition.

In [ ]:
#| default_exp data

## Why one running dataset

Most forecasting tutorials change dataset on every page, so their results cannot be compared. This site uses one dataset for every single-series method, and one for every many-series method. Each notebook then adds a row to the same [scoreboard](../02_Evaluation/03_Scoreboard.ipynb), and the gain of each method is a number, not an impression.

| Dataset | Shape | Used for |
|---|---|---|
| **Household power** (UCI) | 1 house, 1-minute readings, Dec 2006 to Nov 2010, total plus 3 sub-meters | Every single-series forecast, anomaly and change-point notebook |
| **M4 Hourly** | 414 series, 700 or 960 hourly training points each, 48-step test | Global models, hierarchies, benchmarks |

The household data matches the [Phisaver Project](../000_core.ipynb) brief: total consumption of one home, with sub-meters that show which part of the house drew the power. It is public (UCI, CC BY 4.0), so every number on this site can be reproduced. Phisaver's own meter data is private and never appears here.

---

In [ ]:
#| export
import os
import re
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

## Download cache

Files download once into `~/.cache/ml_timeseries` (override with `ML_TIMESERIES_DATA`). The cache sits outside the repo, so no data file is ever committed or published.

---

In [ ]:
#| export
HOUSEHOLD_URL = "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"
M4_TRAIN_URL = "https://auto-arima-results.s3.amazonaws.com/M4-Hourly.csv"
M4_TEST_URL = "https://auto-arima-results.s3.amazonaws.com/M4-Hourly-test.csv"


def cache_dir() -> Path:
    "Download cache: `$ML_TIMESERIES_DATA`, else `~/.cache/ml_timeseries`."
    d = Path(os.environ.get("ML_TIMESERIES_DATA", Path.home() / ".cache" / "ml_timeseries"))
    d.mkdir(parents=True, exist_ok=True)
    return d


def _tmp(p: Path) -> Path:
    # Per-process temp name: nbdev-test runs notebooks in parallel, and several may fill a cold cache at once.
    return p.with_name(f"{p.name}.{os.getpid()}.part")


def fetch(url: str, name: str) -> Path:
    "Download `url` to the cache as `name` unless it is already there."
    p = cache_dir() / name
    if not p.exists():
        tmp = _tmp(p)
        req = urllib.request.Request(url, headers={"User-Agent": "ML_Timeseries-notebooks"})
        with urllib.request.urlopen(req, timeout=120) as r, open(tmp, "wb") as f:
            f.write(r.read())
        os.replace(tmp, p)
    return p


def _save_parquet(df: pd.DataFrame, p: Path):
    tmp = _tmp(p)
    df.to_parquet(tmp)
    os.replace(tmp, p)

## Household power: the raw minutes

The UCI file is a 133 MB semicolon-separated text file inside a 20 MB zip. Missing readings are written as `?`. The first parse takes a few seconds, so the result is cached as Parquet.

| Column | Unit | Meaning |
|---|---|---|
| `active_kw` | kW | Whole-house active power, averaged over the minute |
| `reactive_kw` | kW | Reactive power |
| `voltage` | V | Mains voltage |
| `current_a` | A | Current |
| `sub1_wh` | Wh | Kitchen: dishwasher, oven, microwave |
| `sub2_wh` | Wh | Laundry: washer, dryer, fridge, light |
| `sub3_wh` | Wh | Water heater and air conditioner |

The sub-meters do not cover the whole house. The rest (lighting, sockets, everything else) is the difference between the total and the three sub-meters.

---

In [ ]:
#| export
_RAW_COLS = ["active_kw", "reactive_kw", "voltage", "current_a", "sub1_wh", "sub2_wh", "sub3_wh"]


def load_household_raw() -> pd.DataFrame:
    "One-minute household readings indexed by local timestamp (float32, NaN where missing)."
    p = cache_dir() / "household_minute.parquet"
    if p.exists():
        return pd.read_parquet(p)
    z = fetch(HOUSEHOLD_URL, "household_power_consumption.zip")
    with zipfile.ZipFile(z) as zf, zf.open("household_power_consumption.txt") as f:
        df = pd.read_csv(f, sep=";", na_values="?", low_memory=False)
    ts = pd.to_datetime(df.pop("Date") + " " + df.pop("Time"), format="%d/%m/%Y %H:%M:%S")
    df.columns = _RAW_COLS
    df = df.astype("float32").set_index(pd.DatetimeIndex(ts, name="ts"))
    _save_parquet(df, p)
    return df

In [ ]:
raw = load_household_raw()
print(f"{len(raw):,} minutes from {raw.index[0]} to {raw.index[-1]}")
print(f"missing readings: {raw['active_kw'].isna().mean():.2%}")
raw.head(3).astype(float).round(3).set_axis(raw.index[:3].strftime("%H:%M")).T

2,075,259 minutes from 2006-12-16 17:24:00 to 2010-11-26 21:02:00
missing readings: 1.25%


ts,17:24,17:25,17:26
active_kw,4.216,5.360,5.374
reactive_kw,0.418,0.436,0.498
voltage,234.840,233.630,233.290
current_a,18.400,23.000,23.000
sub1_wh,0.000,0.000,0.000
sub2_wh,1.000,1.000,2.000
sub3_wh,17.000,16.000,17.000


## Household power: energy per period

Forecasting works on energy per period, not instantaneous power. `load_household` resamples the minutes into energy in kWh for each period:

- **Total**: mean kW over the period times the period length in hours.
- **Sub-meters**: the sum of their Wh readings, divided by 1000.
- **Other**: total minus the three sub-meters.

A period with fewer than 75% of its minutes present is set to missing rather than averaged from a fragment. An hour with 5 readings would otherwise look like a full hour.

---

In [ ]:
#| export
def _period_hours(freq: str) -> float:
    t0 = pd.Timestamp("2000-01-03")
    return ((t0 + pd.tseries.frequencies.to_offset(freq)) - t0) / pd.Timedelta(hours=1)


def resample_energy(raw: pd.DataFrame, freq: str = "h", min_coverage: float = 0.75) -> pd.DataFrame:
    "Energy in kWh per `freq` period; periods with too few minutes present become NaN."
    hours = _period_hours(freq)
    g = raw.resample(freq)
    out = pd.DataFrame({"kwh": g["active_kw"].mean() * hours})
    for i in (1, 2, 3):
        out[f"sub{i}_kwh"] = g[f"sub{i}_wh"].sum(min_count=1) / 1000
    out["other_kwh"] = out["kwh"] - out[["sub1_kwh", "sub2_kwh", "sub3_kwh"]].sum(axis=1)
    out["voltage"] = g["voltage"].mean()
    coverage = g["active_kw"].count() / (hours * 60)
    out[coverage < min_coverage] = np.nan
    return out.astype("float64")

## Filling gaps

Most models cannot take missing values, so the loader can fill them, and it marks every filled period in a boolean `filled` column. The [backtest](../02_Evaluation/00_Backtesting.ipynb) uses that flag to score forecasts only against hours the meter actually recorded. The fill rule depends on the gap length:

- **Short gaps (up to 3 periods)**: linear interpolation. Consumption changes little in three hours.
- **Long gaps**: copy the value from one season earlier (168 hours, the same hour last week). Interpolating across two missing days would draw a straight line through two nights and two evenings, which no house ever does.

The [Data Preparation](02_Data_Preparation.ipynb) notebook compares these choices. Here they are only the default.

---

In [ ]:
#| export
def fill_gaps(s: pd.Series, season: int = 168, max_interp: int = 3) -> pd.Series:
    "Interpolate gaps of at most `max_interp` steps; fill longer gaps from `season` steps earlier (or later)."
    s = s.copy()
    isna = s.isna()
    run = (isna != isna.shift()).cumsum()
    run_len = isna.groupby(run).transform("size")
    short = isna & (run_len <= max_interp)
    s[short] = s.interpolate(limit_area="inside")[short]
    for _ in range(100):
        if not s.isna().any():
            break
        s = s.fillna(s.shift(season)).fillna(s.shift(-season))
    return s

In [ ]:
#| export
def load_household(freq: str = "h", clean: bool = True) -> pd.DataFrame:
    "Household energy in kWh per `freq` period. `clean=True` fills every gap (see `fill_gaps`) and adds a `filled` flag."
    p = cache_dir() / f"household_{freq}_{'clean' if clean else 'raw'}_v2.parquet"
    if p.exists():
        return pd.read_parquet(p)
    out = resample_energy(load_household_raw(), freq)
    if clean:
        filled = out["kwh"].isna()
        season = int(round(168 / _period_hours(freq))) if _period_hours(freq) <= 24 else 1
        out = out.apply(fill_gaps, season=max(season, 1))
        out["filled"] = filled
    out.index.freq = freq
    _save_parquet(out, p)
    return out

In [ ]:
hourly_raw = load_household(clean=False)
hourly = load_household()
gaps = hourly_raw["kwh"].isna()
run = (gaps != gaps.shift()).cumsum()
lengths = gaps.groupby(run).sum()
lengths = lengths[lengths > 0]
print(f"{len(hourly):,} hours, {gaps.sum()} missing ({gaps.mean():.2%}) in {len(lengths)} gaps")
print(f"longest gap: {lengths.max()} hours; gaps of 1-3 hours: {(lengths <= 3).sum()}")
hourly.drop(columns="filled").describe().T[["mean", "std", "min", "max"]].round(3)

34,589 hours, 438 missing (1.27%) in 16 gaps
longest gap: 121 hours; gaps of 1-3 hours: 9


,mean,std,min,max
kwh,1.089,0.897,0.124,6.561
sub1_kwh,0.067,0.212,0.000,2.902
sub2_kwh,0.078,0.251,0.000,2.786
sub3_kwh,0.387,0.441,0.000,1.293
other_kwh,0.557,0.495,0.083,3.890
voltage,240.846,2.995,225.835,251.902


## Charts

`line_chart`, `bar_chart` and `show` are the chart helpers the site uses. `line_chart` draws pyecharts lines with the pinned asset host from the repo conventions and a zoom slider, sized to the page width. `show` renders it with the chart data compacted, which keeps a notebook with several charts under a few hundred KB.

---

In [ ]:
#| export
from pyecharts import options as opts
from pyecharts.charts import Line
from pyecharts.globals import CurrentConfig

CurrentConfig.ONLINE_HOST = "https://cdn.jsdelivr.net/gh/pyecharts/pyecharts-assets@f2328d558550046fc5449672a6850e848e141590/assets/v6/"


def line_chart(x, series: dict, y_name: str = "", height: str = "400px", zoom: bool = True):
    "pyecharts line chart of `series` (name -> values) against `x`, sized to the page width."
    c = Line(init_opts=opts.InitOpts(width="100%", height=height))
    c.add_xaxis([str(v) for v in x])
    for name, ys in series.items():
        c.add_yaxis(name, [None if pd.isna(v) else round(float(v), 3) for v in ys],
                    is_symbol_show=False, label_opts=opts.LabelOpts(is_show=False))
    c.set_global_opts(
        tooltip_opts=opts.TooltipOpts(trigger="axis"),
        yaxis_opts=opts.AxisOpts(name=y_name),
        datazoom_opts=[opts.DataZoomOpts(range_start=0, range_end=100)] if zoom else None,
        legend_opts=opts.LegendOpts(pos_top="0"),
    )
    # Room for a legend that wraps to two lines at phone width.
    c.options.update(grid=opts.GridOpts(pos_top="70px", pos_left="50px", pos_right="20px"))
    return c


def bar_chart(labels, series: dict, y_name: str = "", height: str = "400px", horizontal: bool = False):
    "pyecharts bar chart of `series` (name -> values) per label, sized to the page width."
    from pyecharts.charts import Bar
    c = Bar(init_opts=opts.InitOpts(width="100%", height=height))
    c.add_xaxis([str(v) for v in labels])
    for name, ys in series.items():
        c.add_yaxis(name, [None if pd.isna(v) else round(float(v), 4) for v in ys],
                    label_opts=opts.LabelOpts(is_show=False))
    c.set_global_opts(
        tooltip_opts=opts.TooltipOpts(trigger="axis"),
        yaxis_opts=opts.AxisOpts(name=y_name),
        legend_opts=opts.LegendOpts(pos_top="0"),
        xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=0 if horizontal else 30)),
    )
    if horizontal:
        c.reversal_axis()
    c.options.update(grid=opts.GridOpts(pos_top="60px", pos_left="10px", pos_right="30px", pos_bottom="10px", is_contain_label=True))
    return c


def show(chart):
    "Render a pyecharts chart inline with its JSON compacted (pyecharts pretty-prints one value per line)."
    from IPython.display import HTML
    html = chart.render_notebook().data
    html = re.sub(r"(?<=[\[{,:])\s*\n\s*", "", html)
    html = re.sub(r"\s*\n\s*(?=[\]}])", "", html)
    return HTML(html)

### Four years of daily energy

The winter peaks and the August holidays (consumption drops to almost nothing in August 2008) are visible at once. Both matter for every model that follows: the first is annual seasonality, the second is the kind of event no model can predict from the history alone.

In [ ]:
daily = load_household("D")
show(line_chart(daily.index.date, {"total": daily["kwh"], "water heater + AC": daily["sub3_kwh"]},
           y_name="kWh"))

### Two weeks of hourly energy

At hourly resolution the daily cycle dominates: a low night, a morning peak and a larger evening peak. Weekends look different from weekdays. These are the two seasonal periods (24 and 168 hours) that the classical models must capture.

In [ ]:
week = hourly.loc["2009-01-05":"2009-01-18"]
show(line_chart(week.index.strftime("%a %d %H:00"), {"total": week["kwh"], "kitchen": week["sub1_kwh"],
                        "laundry": week["sub2_kwh"], "water heater + AC": week["sub3_kwh"]},
           y_name="kWh"))

## M4 Hourly

The M4 competition (2018) is the standard many-series benchmark. Its hourly subset has 414 series of different scales and patterns, with a fixed 48-hour test horizon. The files use the long format that the Nixtla libraries expect: one row per `(unique_id, ds, y)`, where `ds` is an integer step.

---

In [ ]:
#| export
def load_m4_hourly() -> tuple[pd.DataFrame, pd.DataFrame]:
    "M4 Hourly as long-format `(unique_id, ds, y)` frames: `(train, test)`."
    train = pd.read_csv(fetch(M4_TRAIN_URL, "M4-Hourly.csv"))
    test = pd.read_csv(fetch(M4_TEST_URL, "M4-Hourly-test.csv"))
    return train, test


def to_long(s: pd.Series, uid: str = "house") -> pd.DataFrame:
    "One series as a long-format `(unique_id, ds, y)` frame."
    return pd.DataFrame({"unique_id": uid, "ds": s.index, "y": s.to_numpy()})

In [ ]:
m4_train, m4_test = load_m4_hourly()
lens = m4_train.groupby("unique_id").size()
print(f"{m4_train.unique_id.nunique()} series; train lengths {sorted(int(v) for v in lens.unique())}; "
      f"test {int(m4_test.groupby('unique_id').size().max())} steps each")
m4_train.head(3)

414 series; train lengths [700, 960]; test 48 steps each


,unique_id,ds,y
0,H1,1,605.0
1,H1,2,586.0
2,H1,3,586.0


In [ ]:
#| hide
assert len(raw) == 2_075_259 and list(raw.columns) == _RAW_COLS
assert hourly.index.is_monotonic_increasing and not hourly.index.has_duplicates
assert (hourly.index.to_series().diff().dropna() == pd.Timedelta("1h")).all()
assert not hourly.isna().any().any(), "clean loader left NaN"
assert hourly["filled"].sum() == hourly_raw["kwh"].isna().sum() == 438
assert hourly_raw["kwh"].isna().sum() > 0, "the raw view should keep its gaps"
assert (hourly_raw["other_kwh"].dropna() > -0.05).all(), "sub-meters exceed the total"
assert m4_train.unique_id.nunique() == 414 and set(lens.unique()) == {700, 960}
assert (m4_test.groupby("unique_id").size() == 48).all()
assert to_long(hourly["kwh"]).columns.tolist() == ["unique_id", "ds", "y"]